# MedTime IE Span Baseline Reproduction (Kaggle)

This notebook reproduces the IE Span (BiomedNLP-PubMedBERT) baseline training pipeline in a clean environment.
It uses synthetic data and the same logic as `projects/medtime/baselines.py`.

In [ ]:
!pip install transformers datasets seqeval accelerate -q

In [ ]:
import os
import json
import torch
import torch.nn as nn
import numpy as np
from dataclasses import dataclass
from typing import Optional, List, Dict, Any
from tqdm.auto import tqdm
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForTokenClassification,
    TrainingArguments, Trainer, DataCollatorForTokenClassification,
    EarlyStoppingCallback
)

# --- CONFIG ---
@dataclass
class IESpanConfig:
    lang: str = "en"
    model_path: str = "microsoft/BiomedNLP-PubMedBERT-base-uncased-abstract-fulltext"
    max_seq_len: int = 128
    batch_size: int = 8
    learning_rate: float = 2e-5
    max_steps: int = 50
    warmup_ratio: float = 0.1
    neg_ratio: float = 0.3
    early_stopping_patience: int = 3
    early_stopping_threshold: float = 0.001
    label2id = {"O": 0, "B-EVENT": 1, "I-EVENT": 2}
    id2label = {0: "O", 1: "B-EVENT", 2: "I-EVENT"}

cfg = IESpanConfig()
print(f"Using Model: {cfg.model_path}")

In [ ]:
# --- MOCK DATA GENERATOR ---
def generate_mock_data(n=20):
    data = []
    for i in range(n):
        text = f"Patient {i} experienced severe headache on Jan 2024. Symptoms resolved by Feb 2024."
        # Mock Gold Info
        tl = [
            {"trigger": "headache", "start": 27, "end": 35, "type": "EVENT"},
            {"trigger": "Symptoms", "start": 50, "end": 58, "type": "EVENT"}
        ]
        data.append({
            "pid": f"MOCK_{i}",
            "input": text,
            "meta": json.dumps({"raw_timeline": tl})
        })
    return Dataset.from_list(data)

train_ds_raw = generate_mock_data(20)
dev_ds_raw = generate_mock_data(5)
print(f"Generated {len(train_ds_raw)} train samples")

In [ ]:
# --- DATA PREPARATION ---
tokenizer = AutoTokenizer.from_pretrained(cfg.model_path)

def prepare_bio_dataset(raw_dataset):
    chunked_samples = []
    for ex in tqdm(raw_dataset, desc="Chunking"):
        text = ex["input"]
        try:
            gold_tl = json.loads(ex["meta"])["raw_timeline"]
        except:
            continue

        # Simple sentence splitting by '.'
        sentences = []
        current_pos = 0
        for part in text.split('.'):
            if not part.strip(): continue
            # Reconstruct simplistic offsets (approximate for mock)
            s_start = text.find(part, current_pos)
            if s_start == -1: continue
            s_end = s_start + len(part)
            sentences.append((s_start, s_end, part))
            current_pos = s_end
        
        for s_start, s_end, s_text in sentences:
            char_labels = ["O"] * len(s_text)
            has_evt = False
            for item in gold_tl:
                g_s, g_e = int(item.get("start", -1)), int(item.get("end", -1))
                if s_start <= g_s < s_end:
                    rel_s = g_s - s_start
                    rel_e = min(g_e - s_start, len(s_text))
                    if rel_s < len(char_labels):
                        char_labels[rel_s] = "B-EVENT"
                        for i in range(rel_s + 1, rel_e):
                            char_labels[i] = "I-EVENT"
                        has_evt = True
            
            if not has_evt and np.random.rand() > cfg.neg_ratio:
                continue

            enc = tokenizer(
                s_text,
                truncation=True,
                max_length=cfg.max_seq_len,
                return_offsets_mapping=True
            )
            labels = []
            for s, e in enc["offset_mapping"]:
                if s == e:
                    labels.append(-100)
                else:
                    # Check trigger overlap
                    # Allow loose match for tokenizer splits
                    mid = (s + e) // 2
                    if mid < len(char_labels):
                         labels.append(cfg.label2id.get(char_labels[s], 0))
                    else:
                         labels.append(0)

            chunked_samples.append({
                "input_ids": enc.input_ids,
                "attention_mask": enc.attention_mask,
                "labels": labels
            })
    return Dataset.from_list(chunked_samples)

train_ds = prepare_bio_dataset(train_ds_raw)
dev_ds = prepare_bio_dataset(dev_ds_raw)
print(f"Processed Train: {len(train_ds)}, Dev: {len(dev_ds)}")

In [ ]:
# --- MODEL & TRAINER ---
model = AutoModelForTokenClassification.from_pretrained(cfg.model_path, num_labels=3)

class WeightedIESpanTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        # Class weights 1:5:5 (approx)
        weights = torch.tensor([0.2, 1.0, 1.0]).to(model.device)
        loss_fct = nn.CrossEntropyLoss(weight=weights)
        logits = outputs.get("logits")
        loss = loss_fct(logits.view(-1, 3), labels.view(-1))
        return (loss, outputs) if return_outputs else loss

def compute_metrics(eval_preds):
    logits, labels = eval_preds
    predictions = np.argmax(logits, axis=-1)
    
    true_preds = [
        [cfg.id2label[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [cfg.id2label[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    # Simple Accuracy for smoke test
    correct = sum([1 for p, l in zip(true_preds, true_labels) if p == l])
    total = len(true_preds)
    return {"accuracy": correct / (total + 1e-6)}

training_args = TrainingArguments(
    output_dir="./results",
    evaluation_strategy="steps",
    eval_steps=10,
    save_steps=10,
    logging_steps=5,
    learning_rate=cfg.learning_rate,
    per_device_train_batch_size=cfg.batch_size,
    per_device_eval_batch_size=cfg.batch_size,
    num_train_epochs=3,
    weight_decay=0.01,
    max_steps=cfg.max_steps,
    report_to="none"
)

trainer = WeightedIESpanTrainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    tokenizer=tokenizer,
    data_collator=DataCollatorForTokenClassification(tokenizer),
    compute_metrics=compute_metrics
)

print("🚀 Starting Training Check...")
trainer.train()